# Financial Statement Anomaly Detector

## 1. Project Introduction

### What financial statement anomaly detection means
Financial statement anomaly detection involves identifying unusual patterns or deviations in a company's financial data. It uses ratio analysis, trend analysis, and machine learning to flag potential inconsistencies in how financial performance and position are reported.

### Why it matters in valuation and investment analysis
For equity analysts, valuation professionals, auditors, and forensic accounting reviewers, relying on accurate financial data is paramount. Anomalies could indicate aggressive accounting practices, earnings management, or even fraud. Detecting these early helps adjust valuation models appropriately and mitigates investment risks.

### What types of anomalies the notebook looks for
This notebook specifically looks for:
- Unusual divergence between revenue and cash flow
- Disproportionate growth in working capital items (like receivables and inventory) relative to revenue
- Unexplained margin compressions or expansions
- Sudden increases in debt or weakening liquidity
- High Beneish M-Scores (a probabilistic model for earnings manipulation)
- Statistical outliers compared to peers and historical trends

### The limits of this model
This model relies purely on numerical financial statement data. It does not consider footnotes, management discussions, auditor notes, macroeconomic factors, or qualitative changes in business models. 

### Why the output should be treated as a risk flag, not proof of fraud
**Important:** The output of this notebook is purely diagnostic. Anomalies and statistical outliers are common in rapidly growing companies, companies undergoing restructuring, or those facing macroeconomic shocks. Flags generated by this model should be treated as starting points for deeper fundamental research, **not as proof of fraud or financial manipulation.**


## 2. Import Libraries

We will use standard data science libraries for this analysis. Isolation Forest from `scikit-learn` will be used for unsupervised anomaly detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 3. Create or Load Financial Statement Data

If no live data source is provided, we generate a realistic sample dataset. The dataset includes multiple companies across several years and incorporates a few intentionally abnormal cases to demonstrate the detection capabilities.

**Examples of injected abnormalities:**
- Revenue grows sharply but operating cash flow falls
- Receivables grow much faster than revenue
- Gross margin changes sharply without explanation
- Inventory grows faster than cost of sales
- Debt rises sharply


In [ ]:
def generate_sample_data():
    years = [2019, 2020, 2021, 2022, 2023]
    companies = ['Alpha Corp', 'Beta Ltd', 'Gamma Inc', 'Delta Tech', 'Epsilon Group']
    sectors = {'Alpha Corp': 'Manufacturing', 'Beta Ltd': 'Retail', 'Gamma Inc': 'Tech', 'Delta Tech': 'Tech', 'Epsilon Group': 'Manufacturing'}
    
    data = []
    
    for company in companies:
        base_rev = np.random.randint(500, 1500)
        sector = sectors[company]
        
        for i, year in enumerate(years):
            # Normal growth
            rev_growth = np.random.uniform(1.02, 1.15)
            revenue = base_rev * (rev_growth ** i)
            
            # Base normal metrics
            gross_margin = np.random.uniform(0.3, 0.5)
            operating_margin = gross_margin * np.random.uniform(0.4, 0.7)
            
            gross_profit = revenue * gross_margin
            operating_profit = revenue * operating_margin
            net_income = operating_profit * 0.75 # Assuming 25% tax/interest
            
            total_assets = revenue * np.random.uniform(0.8, 1.5)
            current_assets = total_assets * np.random.uniform(0.4, 0.6)
            cash = current_assets * np.random.uniform(0.1, 0.3)
            accounts_receivable = revenue * np.random.uniform(0.1, 0.2)
            inventory = revenue * np.random.uniform(0.1, 0.2)
            
            total_liabilities = total_assets * np.random.uniform(0.4, 0.7)
            current_liabilities = current_assets * np.random.uniform(0.6, 0.9)
            long_term_debt = total_liabilities - current_liabilities
            equity = total_assets - total_liabilities
            
            operating_cash_flow = net_income * np.random.uniform(0.8, 1.2) + (total_assets * 0.05) # Add back D&A proxy
            capital_expenditure = revenue * np.random.uniform(0.05, 0.1)
            depreciation = total_assets * 0.05
            
            shares_outstanding = 100
            market_cap = net_income * 15 # Simple 15 P/E
            
            row = {
                'company': company,
                'year': year,
                'sector': sector,
                'revenue': revenue,
                'gross_profit': gross_profit,
                'operating_profit': operating_profit,
                'net_income': net_income,
                'total_assets': total_assets,
                'current_assets': current_assets,
                'cash': cash,
                'accounts_receivable': accounts_receivable,
                'inventory': inventory,
                'total_liabilities': total_liabilities,
                'current_liabilities': current_liabilities,
                'long_term_debt': long_term_debt,
                'equity': equity,
                'operating_cash_flow': operating_cash_flow,
                'capital_expenditure': capital_expenditure,
                'depreciation': depreciation,
                'shares_outstanding': shares_outstanding,
                'market_cap': market_cap
            }
            
            # Inject Anomalies
            if company == 'Delta Tech' and year == 2023:
                # Anomaly: Revenue grows sharply but operating cash flow falls. Receivables grow fast.
                row['revenue'] *= 1.5
                row['accounts_receivable'] *= 2.5
                row['operating_cash_flow'] *= 0.2
                row['net_income'] *= 1.4
            
            if company == 'Beta Ltd' and year == 2022:
                # Anomaly: Inventory explodes, gross margin shifts.
                row['inventory'] *= 2.0
                row['gross_profit'] *= 1.3
                
            if company == 'Gamma Inc' and year == 2023:
                # Anomaly: Debt rises sharply, cash flow weakens.
                row['long_term_debt'] *= 3.0
                row['total_liabilities'] = row['current_liabilities'] + row['long_term_debt']
                row['operating_cash_flow'] *= 0.5
                
            data.append(row)
            
    return pd.DataFrame(data)

df = generate_sample_data()
df.head()


## 4. Data Cleaning

Before analysis, we validate the data:
- Check for missing values
- Check for duplicates
- Enforce basic accounting rules (e.g., total assets > 0, revenue >= 0)


In [ ]:
# Missing value checks
if df.isnull().sum().sum() > 0:
    print("Warning: Missing values detected. Filling with 0 or forward filling might be required.")
    df = df.fillna(0) # Simplified handling

# Duplicate checks
df = df.drop_duplicates()

# Data type checks
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Basic validation rules
df = df[df['total_assets'] > 0]
df = df[df['revenue'] >= 0]

# Flag if current assets exceed total assets
df['invalid_assets_flag'] = df['current_assets'] > df['total_assets']
if df['invalid_assets_flag'].any():
    print("Warning: Current assets exceed total assets in some rows. Fixing...")
    df.loc[df['invalid_assets_flag'], 'current_assets'] = df['total_assets']

df = df.drop(columns=['invalid_assets_flag'])
print("Data cleaning complete.")


## 5. Feature Engineering

We calculate standard financial ratios and year-on-year changes to establish baselines and spot trends.


In [ ]:
# Sort by company and year to calculate YoY changes
df = df.sort_values(by=['company', 'year']).reset_index(drop=True)

# Helper for safe division
def safe_div(n, d):
    return np.where(d == 0, 0, n / d)

# Financial Ratios
df['gross_margin'] = safe_div(df['gross_profit'], df['revenue'])
df['operating_margin'] = safe_div(df['operating_profit'], df['revenue'])
df['net_margin'] = safe_div(df['net_income'], df['revenue'])
df['return_on_assets'] = safe_div(df['net_income'], df['total_assets'])
df['return_on_equity'] = safe_div(df['net_income'], df['equity'])
df['current_ratio'] = safe_div(df['current_assets'], df['current_liabilities'])
df['debt_to_equity'] = safe_div(df['long_term_debt'], df['equity'])
df['debt_to_assets'] = safe_div(df['long_term_debt'], df['total_assets'])
df['asset_turnover'] = safe_div(df['revenue'], df['total_assets'])
df['receivables_to_revenue'] = safe_div(df['accounts_receivable'], df['revenue'])
df['inventory_to_revenue'] = safe_div(df['inventory'], df['revenue'])
df['ocf_to_net_income'] = safe_div(df['operating_cash_flow'], df['net_income'])
df['free_cash_flow'] = df['operating_cash_flow'] - df['capital_expenditure']
df['accruals_ratio'] = safe_div((df['net_income'] - df['operating_cash_flow']), df['total_assets'])
df['capex_to_revenue'] = safe_div(df['capital_expenditure'], df['revenue'])
df['depreciation_to_assets'] = safe_div(df['depreciation'], df['total_assets'])

# YoY Changes
grouped = df.groupby('company')
df['rev_growth'] = grouped['revenue'].pct_change()
df['receivables_growth'] = grouped['accounts_receivable'].pct_change()
df['inventory_growth'] = grouped['inventory'].pct_change()
df['ocf_growth'] = grouped['operating_cash_flow'].pct_change()
df['debt_growth'] = grouped['long_term_debt'].pct_change()
df['gross_margin_change'] = grouped['gross_margin'].diff()
df['operating_margin_change'] = grouped['operating_margin'].diff()

# Handle NaNs from pct_change
df = df.fillna(0)


## 6. Beneish M Score

The Beneish M-Score is a mathematical model that uses financial ratios to identify areas where companies might be manipulating their earnings.

**Components:**
- **DSRI (Days Sales in Receivables Index):** Measures whether receivables and revenue are in sync.
- **GMI (Gross Margin Index):** Assesses if gross margins have deteriorated.
- **AQI (Asset Quality Index):** Measures non-current assets other than PP&E.
- **SGI (Sales Growth Index):** High growth may create pressure to manipulate earnings.
- **DEPI (Depreciation Index):** Measures changes in depreciation rate.
- **SGAI (Sales General and Administrative Expenses Index):** Measures SG&A relative to sales.
- **LVGI (Leverage Index):** Measures changes in debt structure.
- **TATA (Total Accruals to Total Assets):** High accruals can indicate earnings management.

*Note: M Score > -2.22 is a warning flag, but not proof of manipulation.*


In [ ]:
def calculate_beneish(df):
    df_b = df.copy()
    
    grouped = df_b.groupby('company')
    
    # Calculate previous year values
    df_b['prev_rev'] = grouped['revenue'].shift(1)
    df_b['prev_rec'] = grouped['accounts_receivable'].shift(1)
    df_b['prev_gp'] = grouped['gross_profit'].shift(1)
    df_b['prev_ta'] = grouped['total_assets'].shift(1)
    df_b['prev_ca'] = grouped['current_assets'].shift(1)
    df_b['prev_dep'] = grouped['depreciation'].shift(1)
    df_b['prev_lt_debt'] = grouped['long_term_debt'].shift(1)
    df_b['prev_cl'] = grouped['current_liabilities'].shift(1)
    
    # Fill NAs to avoid errors, though Beneish is best viewed from year 2 onwards
    df_b = df_b.fillna(1)
    
    # DSRI
    rec_to_rev_t = safe_div(df_b['accounts_receivable'], df_b['revenue'])
    rec_to_rev_t1 = safe_div(df_b['prev_rec'], df_b['prev_rev'])
    df_b['DSRI'] = safe_div(rec_to_rev_t, rec_to_rev_t1)
    
    # GMI
    gm_t = safe_div(df_b['gross_profit'], df_b['revenue'])
    gm_t1 = safe_div(df_b['prev_gp'], df_b['prev_rev'])
    df_b['GMI'] = safe_div(gm_t1, gm_t)
    
    # AQI (Simplified)
    aq_t = 1 - safe_div(df_b['current_assets'], df_b['total_assets'])
    aq_t1 = 1 - safe_div(df_b['prev_ca'], df_b['prev_ta'])
    df_b['AQI'] = safe_div(aq_t, aq_t1)
    
    # SGI
    df_b['SGI'] = safe_div(df_b['revenue'], df_b['prev_rev'])
    
    # DEPI
    dep_rate_t = safe_div(df_b['depreciation'], df_b['total_assets'])
    dep_rate_t1 = safe_div(df_b['prev_dep'], df_b['prev_ta'])
    df_b['DEPI'] = safe_div(dep_rate_t1, dep_rate_t)
    
    # SGAI (Simplified - using operating margin diff as proxy since SG&A isn't explicit)
    df_b['SGAI'] = 1.0 # Defaulting to 1.0 for simplicity with sample data
    
    # LVGI
    lev_t = safe_div((df_b['current_liabilities'] + df_b['long_term_debt']), df_b['total_assets'])
    lev_t1 = safe_div((df_b['prev_cl'] + df_b['prev_lt_debt']), df_b['prev_ta'])
    df_b['LVGI'] = safe_div(lev_t, lev_t1)
    
    # TATA
    df_b['TATA'] = safe_div((df_b['net_income'] - df_b['operating_cash_flow']), df_b['total_assets'])
    
    # Calculate M Score
    df_b['beneish_m_score'] = -4.84 + (0.920 * df_b['DSRI']) + (0.528 * df_b['GMI']) +                               (0.404 * df_b['AQI']) + (0.892 * df_b['SGI']) +                               (0.115 * df_b['DEPI']) - (0.172 * df_b['SGAI']) +                               (4.679 * df_b['TATA']) - (0.327 * df_b['LVGI'])
                              
    df_b['beneish_flag'] = df_b['beneish_m_score'] > -2.22
    
    return df_b

df = calculate_beneish(df)


## 7. Rule-Based Anomaly Detection

We create a scoring system where companies accumulate points based on predefined red flags.

**Risk Levels:**
- **Low:** 0-1 points
- **Medium:** 2-3 points
- **High:** 4+ points


In [ ]:
def assign_anomaly_score(row):
    score = 0
    flags = []
    
    # Receivables growth much faster than revenue
    if row['receivables_growth'] - row['rev_growth'] > 0.20:
        score += 1
        flags.append("Rec > Rev Growth")
        
    # Inventory growth much faster than revenue
    if row['inventory_growth'] - row['rev_growth'] > 0.20:
        score += 1
        flags.append("Inv > Rev Growth")
        
    # Net income positive but operating cash flow negative
    if row['net_income'] > 0 and row['operating_cash_flow'] < 0:
        score += 2
        flags.append("Positive NI, Negative OCF")
        
    # Weak cash conversion
    if row['ocf_to_net_income'] < 0.5:
        score += 1
        flags.append("Weak OCF/NI")
        
    # Sharp gross margin fall
    if row['gross_margin_change'] < -0.10:
        score += 1
        flags.append("Sharp GM Drop")
        
    # Revenue high growth while OCF falls
    if row['rev_growth'] > 0.15 and row['ocf_growth'] < -0.10:
        score += 2
        flags.append("Rev Growth vs OCF Drop")
        
    # Debt spikes
    if row['debt_growth'] > 0.50:
        score += 1
        flags.append("Debt Spike")
        
    # Liquidity risk
    if row['current_ratio'] < 1.0:
        score += 1
        flags.append("Current Ratio < 1")
        
    # High Accruals
    if row['accruals_ratio'] > 0.10:
        score += 1
        flags.append("High Accruals")
        
    # Beneish Flag
    if row['beneish_flag']:
        score += 2
        flags.append("Beneish M-Score > -2.22")
        
    return score, ", ".join(flags)

df['anomaly_score'], df['key_red_flags'] = zip(*df.apply(assign_anomaly_score, axis=1))

def assign_risk_level(score):
    if score >= 4:
        return 'High'
    elif score >= 2:
        return 'Medium'
    else:
        return 'Low'

df['anomaly_risk_level'] = df['anomaly_score'].apply(assign_risk_level)


## 8. Peer Comparison

Companies are compared against their sector peers for the same year to find extreme deviations. We calculate sector medians and flag companies that are far from the norm.


In [ ]:
peer_metrics = ['gross_margin', 'operating_margin', 'receivables_to_revenue', 
                'inventory_to_revenue', 'debt_to_equity', 'accruals_ratio']

# Calculate sector medians per year
sector_medians = df.groupby(['sector', 'year'])[peer_metrics].median().reset_index()
sector_medians = sector_medians.rename(columns={m: f'sector_median_{m}' for m in peer_metrics})

df = pd.merge(df, sector_medians, on=['sector', 'year'], how='left')

# Calculate deviations
for m in peer_metrics:
    df[f'{m}_deviation'] = df[m] - df[f'sector_median_{m}']

# Simple flag if deviation is extreme (e.g., margins > 20% diff from median)
df['peer_anomaly_flag'] = (abs(df['gross_margin_deviation']) > 0.20) | (abs(df['accruals_ratio_deviation']) > 0.10)


## 9. Unsupervised Machine Learning

Because labeled fraud data is exceedingly rare, we use **Isolation Forest**, an unsupervised anomaly detection algorithm. It isolates anomalies instead of profiling normal data points. Financial anomalies often look like extreme outliers across multiple dimensions simultaneously.


In [ ]:
features = ['rev_growth', 'gross_margin', 'operating_margin', 'net_margin', 
            'receivables_to_revenue', 'inventory_to_revenue', 'debt_to_equity', 
            'current_ratio', 'accruals_ratio', 'ocf_to_net_income', 'asset_turnover']

# Handle infinite values and NaNs before ML
ml_df = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
scaled_features = scaler.fit_transform(ml_df)

iso_forest = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
df['ml_outlier_score'] = iso_forest.fit_predict(scaled_features)
# Isolation forest returns -1 for outliers, 1 for inliers
df['isolation_forest_flag'] = df['ml_outlier_score'] == -1


## 10. Visualisation

Visualising the trends helps to contextualise the numerical anomalies.


In [ ]:
plt.style.use('default') # Standard style

def plot_rev_vs_ocf(df, companies):
    plt.figure(figsize=(10, 5))
    for company in companies:
        c_df = df[df['company'] == company]
        plt.plot(c_df['year'], c_df['revenue'], label=f'{company} Rev', linestyle='--')
        plt.plot(c_df['year'], c_df['operating_cash_flow'], label=f'{company} OCF', linewidth=2)
    plt.title('Revenue vs Operating Cash Flow')
    plt.xlabel('Year')
    plt.ylabel('Amount')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('rev_vs_ocf.png')
    plt.show()

# Plot for normal vs anomalous company
plot_rev_vs_ocf(df, ['Alpha Corp', 'Delta Tech'])

# Line trends for Receivables, Inventory, Gross Margin
fig, axes = plt.subplots(3, 1, figsize=(10, 15))
sns.lineplot(data=df, x='year', y='receivables_to_revenue', hue='company', ax=axes[0], marker='o')
axes[0].set_title('Receivables to Revenue Trend')
axes[0].set_ylabel('Receivables / Revenue')

sns.lineplot(data=df, x='year', y='inventory_to_revenue', hue='company', ax=axes[1], marker='o')
axes[1].set_title('Inventory to Revenue Trend')
axes[1].set_ylabel('Inventory / Revenue')

sns.lineplot(data=df, x='year', y='gross_margin', hue='company', ax=axes[2], marker='o')
axes[2].set_title('Gross Margin Trend')
axes[2].set_ylabel('Gross Margin')

for ax in axes:
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ratio_trends.png')
plt.show()

# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='rev_growth', y='ocf_growth', hue='anomaly_risk_level', ax=axes[0])
axes[0].set_title('Revenue Growth vs OCF Growth')
axes[0].set_xlim(-0.5, 1.5)
axes[0].set_ylim(-1.0, 1.5)

sns.scatterplot(data=df, x='accruals_ratio', y='net_margin', hue='isolation_forest_flag', ax=axes[1])
axes[1].set_title('Accruals Ratio vs Net Margin (ML Outliers)')

plt.tight_layout()
plt.savefig('scatter_plots.png')
plt.show()

# Anomaly Score Trend
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='year', y='anomaly_score', hue='company')
plt.title('Anomaly Score by Company and Year')
plt.ylabel('Anomaly Score')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('anomaly_scores.png')
plt.show()

# Beneish M-Score Trend
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='year', y='beneish_m_score', hue='company')
plt.axhline(-2.22, color='red', linestyle='--', label='Warning Threshold (-2.22)')
plt.title('Beneish M-Score by Company and Year')
plt.ylabel('Beneish M-Score')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('beneish_scores.png')
plt.show()


## 11. Company Level Diagnostic Report

A function to generate a plain-English summary of the findings for a specific company.


In [ ]:
def generate_company_report(company_name):
    c_df = df[df['company'] == company_name].sort_values('year')
    if c_df.empty:
        return "Company not found."
    
    latest = c_df.iloc[-1]
    
    print(f"=== Diagnostic Report: {company_name} ===")
    print(f"Years Analysed: {c_df['year'].min()} - {c_df['year'].max()}")
    print(f"Sector: {latest['sector']}")
    print("-" * 40)
    print(f"Latest Anomaly Score: {latest['anomaly_score']}")
    print(f"Latest Risk Level: {latest['anomaly_risk_level']}")
    print(f"Latest Beneish M-Score: {latest['beneish_m_score']:.2f}")
    
    flags = latest['key_red_flags']
    if flags:
        print(f"Key Red Flags Detected: {flags}")
    else:
        print("Key Red Flags Detected: None")
        
    print(f"ML Outlier Flag: {'Yes' if latest['isolation_forest_flag'] else 'No'}")
    print(f"Peer Anomaly Flag: {'Yes' if latest['peer_anomaly_flag'] else 'No'}")
    print("-" * 40)
    
    # Interpretation
    if latest['anomaly_risk_level'] == 'High' or latest['isolation_forest_flag'] or latest['beneish_flag']:
        print("Interpretation: This company shows signs that may deserve further review.")
        print("There are notable divergences in financial trends or peer comparisons.")
    elif latest['anomaly_risk_level'] == 'Medium':
        print("Interpretation: Moderate unusual trends detected. Standard fundamental review recommended.")
    else:
        print("Interpretation: No significant structural anomalies detected based on the provided metrics.")
    print("=" * 40)

# Generate a report for our known anomalous entity
generate_company_report('Delta Tech')


## 12. Final Ranked Table

A summary table sorting companies by their risk level, allowing analysts to prioritize their review.


In [ ]:
final_cols = ['company', 'year', 'sector', 'anomaly_score', 'anomaly_risk_level', 
              'beneish_m_score', 'beneish_flag', 'isolation_forest_flag', 'key_red_flags']

final_table = df[final_cols].sort_values(by=['anomaly_score', 'beneish_m_score'], ascending=[False, False])
display(final_table.head(10))


## 13. Export Outputs


In [ ]:
final_table.to_csv('final_anomaly_results.csv', index=False)
print("Saved final results to 'final_anomaly_results.csv'")

# Save diagnostic report for top risk entity
top_risk_company = final_table.iloc[0]['company']
top_risk_df = df[df['company'] == top_risk_company]
top_risk_df.to_csv(f'{top_risk_company}_diagnostic_data.csv', index=False)
print(f"Saved specific data for {top_risk_company}")


## 14. Documentation

### Project Overview
The Financial Statement Anomaly Detector is a robust diagnostic tool built to highlight unusual trends in corporate financial statements. It uses ratio analysis, the Beneish M-Score, peer comparisons, and Unsupervised Machine Learning (Isolation Forests) to score and rank financial data.

### How to Use
1. Run all cells sequentially.
2. Review the generated charts for visual cues.
3. Check the Final Ranked Table to prioritize companies for deeper review.
4. Use the `generate_company_report('CompanyName')` function for a quick summary.

### How to Replace Sample Data with Real Data
Modify the code in Section 3 to load your own data:
```python
# df = pd.read_csv('your_financial_data.csv')
```
Ensure your CSV has the required columns listed in the initial dataset generation step (e.g., revenue, net_income, total_assets).

### Limitations
- The model ignores text-based disclosures, footnotes, and qualitative business changes.
- High growth companies or companies doing major acquisitions often trigger false positives.
- A high risk score **does not** confirm fraud.

## 15. Future Improvements
To build upon this notebook, consider adding:
- **Real Annual Report Data:** Connect to SEC EDGAR, Yahoo Finance, or Capital IQ APIs.
- **Add quarterly data:** To detect seasonality manipulations.
- **Add auditor changes & related party transaction flags:** Crucial qualitative indicators.
- **Add NLP analysis:** Sentiment analysis on Management Discussion and Analysis (MD&A) sections.
- **Add sector specific models:** Banks and Insurers require entirely different ratio sets.
- **Add SHAP explainability:** To explain exactly why the Isolation Forest flagged an entity.
- **Dashboarding:** Wrap this notebook into a Streamlit web application.
- **Database support:** Store historical calculations in a SQL database.
